# Environment-aware transportability

This notebook asks whether a new trial year becomes more predictable when we represent the environment explicitly. It keeps **prospective coarse weather** separate from an **in-season untreated-control sentinel**, then uses leave-one-year-out validation as a gate before promoting product × environment interactions.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from crop_protection_ps.environment_demo import run_environment_demo
summary = run_environment_demo(ROOT)
summary

{'release': '0.4.0',
 'environment_data': {'trial_location': 'experimental hop yard near Corvallis, Oregon',
  'weather_resolution': 'monthly Corvallis-area summaries',
  'weather_months': ['February', 'March', 'April', 'May'],
  'external_climatology_years': [2009,
   2010,
   2011,
   2012,
   2013,
   2014,
   2015,
   2016],
  'primary_trial_years': [2017, 2018, 2020, 2021],
  'limitation': 'Weather is a coarse geographic/monthly proxy and is not treated as exact plot exposure.'},
 'models': {'ridge_alpha': 10.0,
  'baseline': 'treatment + timing',
  'coarse_weather': 'treatment + timing + pre-season wetness + application-window wetness + application-window temperature anomaly',
  'sentinel': 'treatment + timing + same-year untreated-control disease pressure',
  'sentinel_gxe': 'sentinel model + timing-by-sentinel and product-by-sentinel interactions'},
 'leave_one_year_out': {'baseline_rmse': 8.53148587796627,
  'coarse_weather_rmse': 20.21416591841774,
  'sentinel_rmse': 5.176149

## Environment state

Weather anomalies are defined relative to the fixed 2009–2016 Corvallis climatology. The untreated-control sentinel is computed from the five `NT` plots in each trial year and is therefore an in-season state signal.

In [2]:
environment = pd.read_csv(ROOT / "results/real_hop_trial/environment/environment_state_by_year.csv")
environment

,year,preseason_wetness_z,application_wetness_z,application_avg_temp_c_z,sentinel_mean_audpc,sentinel_sqrt_audpc,sentinel_n
0,2017,2.386652,0.489404,0.233882,243.386,15.600833,5
1,2018,-0.308404,-0.349264,0.586913,188.048,13.713059,5
2,2020,-1.359346,0.056681,0.939943,909.146,30.152048,5
3,2021,0.141063,-1.663941,0.939943,941.734,30.687685,5


## Genuine leave-one-year-out comparison

Every treated observation is predicted by a model fitted without its trial year. The same-year sentinel model may use the held-out year's untreated controls, but never its treated outcomes.

In [3]:
metrics = pd.read_csv(ROOT / "results/real_hop_trial/environment/leave_one_year_out_model_metrics.csv")
metrics.loc[metrics["held_out_year"].astype(str) == "ALL"]

,model,held_out_year,n_test,rmse,mae,r2
4,baseline,ALL,187,8.531486,7.526642,-0.657971
9,coarse_weather,ALL,187,20.214166,18.400126,-8.307633
14,sentinel,ALL,187,5.176150,4.312476,0.389703
19,sentinel_gxe,ALL,187,5.252286,4.359107,0.371617


In [4]:
folds = metrics.loc[metrics["held_out_year"].astype(str) != "ALL"].copy()
folds.pivot(index="held_out_year", columns="model", values="rmse")

model,baseline,coarse_weather,sentinel,sentinel_gxe
held_out_year,,,,
2017,4.833854,32.188653,3.545933,3.571077
2018,9.359820,9.123942,3.184292,3.247282
2020,11.820323,19.565233,7.199475,7.246711
2021,5.253890,17.188518,5.378420,5.538679


The important result is not that “weather does not matter.” The available monthly Corvallis summaries are a weak environment representation for only four independent trial years. In contrast, the untreated sentinel measures realised disease pressure directly and cuts aggregate unseen-year RMSE by about 39%.

## Evidence gate for G×E complexity

Product-specific environment slopes are only promoted if they improve aggregate unseen-year RMSE **and** beat the simpler sentinel model in at least three of four held-out years.

In [5]:
gate = json.loads((ROOT / "results/real_hop_trial/environment/gxe_promotion_gate.json").read_text())
gate

{'candidate_model': 'sentinel_gxe',
 'reference_model': 'sentinel',
 'aggregate_rmse_candidate': 5.252285899695607,
 'aggregate_rmse_reference': 5.176149641938232,
 'fold_wins': 0,
 'n_folds': 4,
 'promoted': False}

The richer interaction model fails the gate. That is retained as a result: with four independent environments, a plausible G×E story is not enough to justify a more complex predictive surface.

## Regularisation sensitivity

In [6]:
sensitivity = pd.read_csv(ROOT / "results/real_hop_trial/environment/regularisation_sensitivity.csv")
sensitivity.pivot(index="alpha", columns="model", values="rmse")

model,baseline,coarse_weather,sentinel,sentinel_gxe
alpha,,,,
1.0,8.573039,29.197253,5.189022,5.398923
5.0,8.535909,23.414963,5.155801,5.232668
10.0,8.531486,20.214166,5.176150,5.252286
20.0,8.597747,17.504892,5.339365,5.416377


Across the fixed penalty grid, the qualitative conclusion is stable: the sentinel representation is much more useful than coarse weather, while the interaction extension does not produce a reliable transportability gain.